# ViFinQA — Kaggle GPU: Qwen3-8B-AWQ grounded generation

1. Chọn GPU (T4 x2 dùng được) và bật Internet; attach `vifinqa`, `vifinqa-artifacts`, cùng dense artifact đã tạo bởi notebook build.
2. Nếu GitHub repo còn private, tạo Kaggle Secret tên `GITHUB_TOKEN` có quyền read-only Contents và bật quyền dùng secret cho notebook.
3. Chạy từ trên xuống đến smoke. Chỉ khi 5 dự đoán/traces không lỗi mới đổi cả `VIFINQA_FINAL_RUN` và `VIFINQA_RUN_FULL` thành `1`, chạy lại cell cấu hình rồi chạy cell full.

Kaggle tự giải nén ZIP dataset. Smoke dùng 5 ID cố định đại diện cho nhiều đơn vị và độ phức tạp; full run resume cùng checkpoint.

In [ ]:
# Immutable revisions verified before the competition cutoff. Edit only the two run flags below.
import os

os.environ["VIFINQA_MODEL"] = "Qwen/Qwen3-8B-AWQ"
os.environ["VIFINQA_MODEL_REVISION"] = "4da05a8edb55c6046cce958586c33b61da07bb79"
os.environ["VIFINQA_DENSE_REVISION"] = "5617a9f61b028005a4858fdac845db406aefb181"
os.environ["VIFINQA_THINKING_MODE"] = "disabled"
os.environ["VIFINQA_FINAL_RUN"] = "0"
os.environ["VIFINQA_RUN_FULL"] = "0"
os.environ["VIFINQA_TP"] = "1"
os.environ["VIFINQA_DP"] = "2"
print(
    "model/final/full/tp/dp:",
    os.environ["VIFINQA_MODEL"],
    os.environ["VIFINQA_FINAL_RUN"],
    os.environ["VIFINQA_RUN_FULL"],
    os.environ["VIFINQA_TP"],
    os.environ["VIFINQA_DP"],
)

In [ ]:
import base64
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import requests
import torch

try:
    from kaggle_secrets import UserSecretsClient

    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    del hf_token
    print("authenticated Hugging Face downloads enabled")

print(
    "torch", torch.__version__, "cuda", torch.cuda.is_available(), "gpus", torch.cuda.device_count()
)
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, round(p.total_memory / 2**30, 1), "GiB")
assert torch.cuda.is_available(), "Enable a GPU accelerator before continuing."
FINAL_RUN = os.environ.get("VIFINQA_FINAL_RUN") == "1"
RUN_FULL = os.environ.get("VIFINQA_RUN_FULL") == "1"
MODEL = os.environ["VIFINQA_MODEL"]
MODEL_REVISION = os.environ.get("VIFINQA_MODEL_REVISION")
DENSE_REVISION = os.environ.get("VIFINQA_DENSE_REVISION")
THINKING_MODE = os.environ.get("VIFINQA_THINKING_MODE", "disabled")
assert THINKING_MODE in {"disabled", "auto"}
if FINAL_RUN:
    for name, revision in {"model": MODEL_REVISION, "dense": DENSE_REVISION}.items():
        valid_revision = (
            revision and len(revision) == 40 and all(c in "0123456789abcdef" for c in revision)
        )
        assert valid_revision, f"Final run requires a full lowercase commit SHA for {name}."

In [ ]:
# Prefer attached code. Otherwise clone anonymously, then use a Kaggle GITHUB_TOKEN secret.
GIT_URL = "https://github.com/ThanhDatVN/AI-Financial-Data-Assistant.git"
PROJECT = Path("/kaggle/working/AI-Financial-Data-Assistant")
assert PROJECT.parent == Path("/kaggle/working")


def remove_partial_checkout() -> None:
    if PROJECT.exists() and not (PROJECT / "pyproject.toml").exists():
        shutil.rmtree(PROJECT)


def clone_repo() -> None:
    result = subprocess.run(
        ["git", "clone", "--depth", "1", GIT_URL, str(PROJECT)],
        capture_output=True,
        text=True,
    )
    if result.returncode == 0:
        return
    remove_partial_checkout()
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        token = None
    if not token:
        detail = (result.stderr or "git clone failed").strip().splitlines()[-1]
        raise RuntimeError(
            f"{detail} Enable Internet and either make the repo public, attach the code as a "
            "Kaggle Dataset, or add a read-only GITHUB_TOKEN under Add-ons > Secrets."
        )
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env = os.environ.copy()
    git_env.update(
        {
            "GIT_CONFIG_COUNT": "1",
            "GIT_CONFIG_KEY_0": "http.extraHeader",
            "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}",
        }
    )
    authenticated = subprocess.run(
        ["git", "clone", "--depth", "1", GIT_URL, str(PROJECT)],
        env=git_env,
        capture_output=True,
        text=True,
    )
    del token, auth, git_env
    if authenticated.returncode != 0:
        remove_partial_checkout()
        raise RuntimeError("Authenticated clone failed; verify the read-only GITHUB_TOKEN.")


remove_partial_checkout()
attached_candidates = sorted(
    {
        marker.parent
        for marker in Path("/kaggle/input").rglob("pyproject.toml")
        if (marker.parent / "scripts/50_generate_programs.py").is_file()
    },
    key=str,
)
if not PROJECT.exists() and attached_candidates:
    shutil.copytree(attached_candidates[0], PROJECT)
elif not PROJECT.exists():
    clone_repo()
assert (PROJECT / "pyproject.toml").exists(), f"Invalid project checkout: {PROJECT}"

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements-gpu.txt")],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT)], check=True)
subprocess.run(
    [
        sys.executable,
        "-c",
        "import bm25s, faiss, ftfy, lxml, openai, pandas, pyarrow, rapidfuzz; "
        "import sentence_transformers, Stemmer, unidecode",
    ],
    check=True,
)
package_probe = (
    "import importlib.metadata as m; import torch; "
    "print('resolved runtime:', 'torch', torch.__version__, 'vllm', m.version('vllm'), "
    "'sentence-transformers', m.version('sentence-transformers'), "
    "'transformers', m.version('transformers'), 'openai', m.version('openai')); "
    "assert torch.__version__.startswith('2.10.'); "
    "assert m.version('vllm') == '0.19.1'; "
    "assert m.version('sentence-transformers') == '5.5.1'; "
    "assert m.version('transformers') == '5.5.3'; "
    "assert m.version('openai').split('.')[0] == '2'"
)
subprocess.run([sys.executable, "-c", package_probe], check=True)
os.chdir(PROJECT)
PROJECT_SHA = (
    subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    if (PROJECT / ".git").exists()
    else "attached-archive-no-git-sha"
)
if FINAL_RUN:
    assert len(PROJECT_SHA) == 40, "Final run must use a Git checkout with a recorded SHA."
print("project revision:", PROJECT_SHA)
RUNTIME_LOG = Path("/kaggle/working/runtime_environment.txt")
RUNTIME_LOG.write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True),
    encoding="utf-8",
)

In [ ]:
# Discover inputs by their contents, because Kaggle dataset slugs and nesting can vary.
INPUT_ROOT = Path("/kaggle/input")
mounted_inputs = sorted(path.name for path in INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []
print("Kaggle input mounts:", mounted_inputs or "<none>")
data_candidates = sorted(
    {
        questions.parent.parent
        for questions in INPUT_ROOT.rglob("questions.jsonl")
        if questions.parent.name == "questions"
        and (questions.parent.parent / "code_stock.csv").is_file()
        and (questions.parent.parent / "financial_statements").is_dir()
    },
    key=str,
)
assert data_candidates, (
    "ViFinQA files are not mounted in this Kaggle session. "
    "Attach the `vifinqa` input and restart the session if needed. "
    f"Visible inputs: {mounted_inputs}"
)
DATA_ROOT = data_candidates[0]
manifest_candidates = sorted(
    {
        manifest
        for manifest in INPUT_ROOT.rglob("table_manifest.jsonl")
        if manifest.parent.name == "processed" and manifest.with_suffix(".parquet").is_file()
    },
    key=str,
)
assert manifest_candidates, (
    "Frozen retrieval artifacts are not mounted. "
    f"Attach the `vifinqa-artifacts` input. Visible inputs: {mounted_inputs}"
)
MANIFEST = manifest_candidates[0]
ARTIFACT_INPUT = MANIFEST.parents[2]
BM25 = ARTIFACT_INPUT / "data/index/bm25"
assert MANIFEST.with_suffix(".parquet").exists(), "The generation stage needs the Parquet manifest."
assert BM25.exists(), "Attach or build the BM25 index."


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


expected_hashes = {
    MANIFEST: "ced1d671d6a71c299fea02d7d12b86b596b430a2714ab9aeaa4b338fe012fac1",
    MANIFEST.with_suffix(".parquet"): (
        "060bd26eff14d30ce70b3ba7b00af509be6100b58ddb5a0fe970afa0ef69e29d"
    ),
    MANIFEST.with_suffix(".metadata.json"): (
        "bb338c8a01381241a915517fe774b045cc53213eddd4194f645473616c188e12"
    ),
    BM25 / "data.csc.index.npy": "3f2d7292960e8fda6ca5a1f09d10f692259629ac66cc61703b275b927d5cd683",
    BM25 / "indices.csc.index.npy": (
        "6f5ac7fa7be96f946eced6dcfa7eddeb63493d317f13ef4249bc287e3d993991"
    ),
    BM25 / "indptr.csc.index.npy": (
        "6c136f30e633641b65b26ba2341a0a0aedbba2932aa1d51b4b48e953fbf247eb"
    ),
    BM25 / "params.index.json": "b42a70b508494aa5bb40ef81323a35b4b4ed5afbb7984ed224bb700197a01e7c",
    BM25 / "records.jsonl": "1b8ebe896b92e77c71e5ebadb2e519b377932e1b5a9665080e457fce12daf40f",
    BM25 / "vocab.index.json": "3c5482d9e193cb4240e329de6406779ee5a67f1b4f570bde0d4c397aa297f5aa",
}
for path, expected in expected_hashes.items():
    assert path.exists(), f"Missing frozen artefact: {path}"
    actual = sha256(path)
    assert actual == expected, f"SHA-256 mismatch for {path}: {actual}"
question_count = sum(
    1 for line in (DATA_ROOT / "questions/questions.jsonl").open(encoding="utf-8") if line.strip()
)
assert question_count == 1012, f"Expected 1,012 questions, found {question_count}"
print("inputs verified:", DATA_ROOT, MANIFEST, BM25, "questions=", question_count)

In [ ]:
# Reuse the immutable dense Dataset produced by 01_kaggle_build_dense_artifact.ipynb.
dense_artifacts = []
for artifact_path in INPUT_ROOT.rglob("artifact_manifest.json"):
    try:
        metadata = json.loads(artifact_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        continue
    if (
        metadata.get("artifact_type") == "vifinqa_bge_m3_dense_index"
        and metadata.get("model_revision") == DENSE_REVISION
        and metadata.get("tables") == 146246
    ):
        dense_artifacts.append((artifact_path.parent, metadata))
assert dense_artifacts, (
    "Attach the `vifinqa-dense-bge-m3` Dataset produced by the build notebook; "
    "generation intentionally refuses to rebuild the 146,246-table index."
)
DENSE_ARTIFACT_ROOT, dense_artifact = sorted(dense_artifacts, key=lambda item: str(item[0]))[0]
DENSE = DENSE_ARTIFACT_ROOT / "data/index/bge_m3"
assert dense_artifact.get("source_manifest_sha256") == expected_hashes[MANIFEST]
for relative, file_metadata in dense_artifact.get("files", {}).items():
    artifact_file = DENSE_ARTIFACT_ROOT / relative
    assert artifact_file.exists(), f"Missing dense artifact file: {artifact_file}"
    assert sha256(artifact_file) == file_metadata["sha256"], artifact_file
dense_config = json.loads((DENSE / "config.json").read_text(encoding="utf-8"))
assert (
    dense_config.get("tables") == 146246
), f"Dense index must contain 146,246 tables: {dense_config}"
assert dense_config.get("max_seq_length") == 8192, dense_config
assert dense_config.get("use_fp16") is False, dense_config
assert dense_config.get("model_revision") == DENSE_REVISION, dense_config
gpu_free_bytes = [torch.cuda.mem_get_info(index)[0] for index in range(torch.cuda.device_count())]
DENSE_GPU = max(range(len(gpu_free_bytes)), key=gpu_free_bytes.__getitem__)
assert gpu_free_bytes[DENSE_GPU] / 2**30 >= 3, "Dense query encoding needs 3 GiB free."
DENSE_QUERY_DEVICE = f"cuda:{DENSE_GPU}"
print("reusing verified dense artifact:", DENSE, "query device:", DENSE_QUERY_DEVICE)
RETRIEVAL = Path("/kaggle/working/artifacts/retrieval_hybrid.jsonl")
subprocess.run(
    [
        sys.executable,
        "scripts/30_retrieve_questions.py",
        "--questions",
        str(DATA_ROOT / "questions/questions.jsonl"),
        "--companies",
        str(DATA_ROOT / "code_stock.csv"),
        "--bm25",
        str(BM25),
        "--dense",
        str(DENSE),
        "--dense-device",
        DENSE_QUERY_DEVICE,
        "--output",
        str(RETRIEVAL),
        "--candidate-k",
        "2000",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "scripts/32_validate_retrieval.py",
        "--questions",
        str(DATA_ROOT / "questions/questions.jsonl"),
        "--manifest",
        str(MANIFEST.with_suffix(".parquet")),
        "--retrieval",
        str(RETRIEVAL),
        "--output",
        "/kaggle/working/artifacts/retrieval_hybrid_qc.json",
    ],
    check=True,
)
retrieval_qc_path = Path("/kaggle/working/artifacts/retrieval_hybrid_qc.json")
retrieval_qc = json.loads(retrieval_qc_path.read_text(encoding="utf-8"))
assert retrieval_qc.get("passed") is True and retrieval_qc.get("rows") == 1012, retrieval_qc
print("hybrid retrieval verified:", retrieval_qc_path, retrieval_qc["retrieval_sha256"])

In [ ]:
# Replicate Qwen3-8B-AWQ once per T4; full generation sends two independent shards.
TP = int(os.environ.get("VIFINQA_TP", "1"))
DP = int(os.environ.get("VIFINQA_DP", "2"))
assert TP == 1, "Qwen3-8B-AWQ fits one T4; keep tensor parallelism at 1."
assert 1 <= DP <= torch.cuda.device_count(), "VIFINQA_DP must not exceed the GPU count."
VLLM_BASE = "http://127.0.0.1:8000"
VLLM_CONFIG = Path("/kaggle/working/vllm_server_config.json")
expected_server_config = {
    "model": MODEL,
    "model_revision": MODEL_REVISION,
    "tensor_parallel_size": TP,
    "data_parallel_size": DP,
}


def served_model_ids() -> set[str]:
    try:
        health = requests.get(f"{VLLM_BASE}/health", timeout=2)
        if not health.ok:
            return set()
        response = requests.get(f"{VLLM_BASE}/v1/models", timeout=5)
        response.raise_for_status()
        return {item["id"] for item in response.json().get("data", [])}
    except (requests.RequestException, KeyError, TypeError, ValueError):
        return set()


existing_models = served_model_ids()
if existing_models:
    assert (
        MODEL in existing_models
    ), f"Port 8000 already serves {sorted(existing_models)}, not {MODEL}. Restart the session."
    assert VLLM_CONFIG.exists(), (
        "A pre-existing vLLM server has unknown TP/DP; restart the session."
    )
    actual_server_config = json.loads(VLLM_CONFIG.read_text(encoding="utf-8"))
    assert (
        actual_server_config == expected_server_config
    ), f"Existing vLLM config {actual_server_config} != {expected_server_config}; restart."
    print("reusing healthy vLLM server:", sorted(existing_models))
else:
    server_log = open("/kaggle/working/vllm.log", "a", encoding="utf-8")  # noqa: SIM115
    serve_cmd = [
        "vllm",
        "serve",
        MODEL,
        "--served-model-name",
        MODEL,
        "--host",
        "127.0.0.1",
        "--port",
        "8000",
        "--tensor-parallel-size",
        str(TP),
        "--data-parallel-size",
        str(DP),
        "--api-server-count",
        "1",
        "--dtype",
        "half",
        "--quantization",
        "awq",
        "--max-model-len",
        "16384",
        "--max-num-seqs",
        "1",
        "--gpu-memory-utilization",
        "0.90",
        "--generation-config",
        "vllm",
        "--default-chat-template-kwargs",
        '{"enable_thinking": false}',
        "--seed",
        "20260802",
    ]
    if MODEL_REVISION:
        serve_cmd += ["--revision", MODEL_REVISION]
    server = subprocess.Popen(serve_cmd, stdout=server_log, stderr=subprocess.STDOUT)

    for _ in range(120):
        if MODEL in served_model_ids():
            break
        if server.poll() is not None:
            raise RuntimeError(Path("/kaggle/working/vllm.log").read_text()[-4000:])
        time.sleep(5)
    else:
        raise TimeoutError("vLLM did not become healthy; inspect /kaggle/working/vllm.log")
    VLLM_CONFIG.write_text(
        json.dumps(expected_server_config, indent=2) + "\n", encoding="utf-8"
    )
    print("vLLM ready:", MODEL, MODEL_REVISION, "tp/dp=", TP, DP)

In [ ]:
# Smoke five fixed cases: simple, USD, multi-company, percentage, and trillion-VND.
SMOKE_GEN = Path("/kaggle/working/artifacts/generation_qwen3_8b_awq_smoke")
SMOKE_IDS = [1, 213, 399, 442, 473]
smoke_cmd = [
    sys.executable,
    "scripts/50_generate_programs.py",
    "--retrieval",
    str(RETRIEVAL),
    "--manifest",
    str(MANIFEST.with_suffix(".parquet")),
    "--data-root",
    str(DATA_ROOT),
    "--output",
    str(SMOKE_GEN),
    "--model",
    MODEL,
    "--thinking-mode",
    THINKING_MODE,
    "--max-attempts",
    "3",
    "--project-revision",
    PROJECT_SHA,
]
for question_id in SMOKE_IDS:
    smoke_cmd += ["--id", str(question_id)]
if MODEL_REVISION:
    smoke_cmd += ["--model-revision", MODEL_REVISION]
if FINAL_RUN:
    smoke_cmd += ["--final-run"]
subprocess.run(smoke_cmd, check=True)
smoke_errors = []
if (SMOKE_GEN / "errors.jsonl").exists():
    smoke_errors = [
        json.loads(line) for line in (SMOKE_GEN / "errors.jsonl").read_text().splitlines() if line
    ]
smoke_predictions = json.loads((SMOKE_GEN / "submission.json").read_text(encoding="utf-8"))
smoke_traces = []
if (SMOKE_GEN / "program_traces.jsonl").exists():
    smoke_traces = [
        json.loads(line)
        for line in (SMOKE_GEN / "program_traces.jsonl").read_text(encoding="utf-8").splitlines()
        if line
    ]
print(
    "smoke predictions/errors/traces:", len(smoke_predictions), len(smoke_errors), len(smoke_traces)
)
print(json.dumps(smoke_predictions[:2], ensure_ascii=False, indent=2)[:4000])
assert not smoke_errors, "Smoke recorded errors; inspect errors.jsonl before continuing."
assert {row["id"] for row in smoke_predictions} == set(SMOKE_IDS)
assert {row["id"] for row in smoke_traces} == set(SMOKE_IDS)

In [ ]:
# Full resume-safe final generation. Enable both flags in config, rerun config, then this cell.
RUN_FULL = os.environ.get("VIFINQA_RUN_FULL") == "1"
FINAL_RUN = os.environ.get("VIFINQA_FINAL_RUN") == "1"
assert RUN_FULL, "Set VIFINQA_RUN_FULL=1 only after inspecting smoke output."
assert FINAL_RUN, "Set VIFINQA_FINAL_RUN=1 for the pinned submission-candidate run."
assert THINKING_MODE == "disabled", "Final run requires Qwen3 non-thinking mode."
assert len(PROJECT_SHA) == 40, "Final run requires a recorded project Git SHA."
dense_config = json.loads((DENSE / "config.json").read_text(encoding="utf-8"))
assert (
    dense_config.get("model_revision") == DENSE_REVISION
), "Dense index was not built with the approved BGE-M3 revision."
GEN = Path("/kaggle/working/artifacts/generation_qwen3_8b_awq")
GEN_SHARDS = Path("/kaggle/working/artifacts/generation_qwen3_8b_awq_shards")
prior_generation = sorted(
    INPUT_ROOT.rglob("generation_qwen3_8b_awq_shards/shard_0/run_metadata.json"),
    key=str,
)
if not GEN_SHARDS.exists() and prior_generation:
    prior_shards = prior_generation[0].parents[1]
    shutil.copytree(prior_shards, GEN_SHARDS)
    print("imported prior generation checkpoints:", prior_shards)
common_cmd = [
    sys.executable,
    "scripts/50_generate_programs.py",
    "--retrieval",
    str(RETRIEVAL),
    "--manifest",
    str(MANIFEST.with_suffix(".parquet")),
    "--data-root",
    str(DATA_ROOT),
    "--model",
    MODEL,
    "--thinking-mode",
    THINKING_MODE,
    "--max-attempts",
    "3",
    "--project-revision",
    PROJECT_SHA,
]
if MODEL_REVISION:
    common_cmd += ["--model-revision", MODEL_REVISION]
common_cmd += ["--final-run"]
shard_dirs = [GEN_SHARDS / f"shard_{index}" for index in range(DP)]
workers = []
for shard_index, shard_dir in enumerate(shard_dirs):
    shard_cmd = common_cmd + [
        "--output",
        str(shard_dir),
        "--shard-count",
        str(DP),
        "--shard-index",
        str(shard_index),
    ]
    workers.append((shard_index, subprocess.Popen(shard_cmd)))
for shard_index, worker in workers:
    return_code = worker.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, ["generation-shard", str(shard_index)])
subprocess.run(
    [
        sys.executable,
        "scripts/51_merge_generation_shards.py",
        *[str(path) for path in shard_dirs],
        "--output",
        str(GEN),
        "--expected-rows",
        "1012",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "scripts/40_validate_submission.py",
        str(GEN / "submission.json"),
        "--questions",
        str(DATA_ROOT / "questions/questions.jsonl"),
        "--evidence-root",
        str(GEN),
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "scripts/41_package_submission.py",
        str(GEN / "submission.json"),
        "/kaggle/working/submission.zip",
        "--questions",
        str(DATA_ROOT / "questions/questions.jsonl"),
        "--evidence-root",
        str(GEN),
    ],
    check=True,
)
submission_zip = Path("/kaggle/working/submission.zip")
final_predictions = json.loads((GEN / "submission.json").read_text(encoding="utf-8"))
final_errors = []
if (GEN / "errors.jsonl").exists():
    final_errors = [
        line for line in (GEN / "errors.jsonl").read_text(encoding="utf-8").splitlines() if line
    ]
assert len(final_predictions) == 1012 and not final_errors
FINAL_MANIFEST = Path("/kaggle/working/final_artifacts.json")
final_manifest = {
    "project_revision": PROJECT_SHA,
    "model": MODEL,
    "model_revision": MODEL_REVISION,
    "dense_revision": DENSE_REVISION,
    "thinking_mode": THINKING_MODE,
    "tensor_parallel_size": TP,
    "data_parallel_size": DP,
    "retrieval_sha256": sha256(RETRIEVAL),
    "submission_json_sha256": sha256(GEN / "submission.json"),
    "submission_zip_sha256": sha256(submission_zip),
    "runtime_environment_sha256": sha256(RUNTIME_LOG),
    "questions": len(final_predictions),
}
FINAL_MANIFEST.write_text(
    json.dumps(final_manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print(submission_zip, submission_zip.stat().st_size, final_manifest["submission_zip_sha256"])
print(FINAL_MANIFEST, json.dumps(final_manifest, indent=2))